# Wild Apple Forest Local Python Workflow

Phase-B scaffold for the Ili 2021 wild apple forest workflow.

Current scope:
- migrate the core logic of `ili_demo3_debug.js` into local GEE Python
- switch the image source to Sentinel-2 + Sentinel-1 monthly composites
- keep the structure aligned with the three project documents in this repo
- provide a first runnable baseline for sample extraction and local RF modeling

Later steps will fill in:
- sample cleaning rules
- cubic time-series fitting
- Jeffries-Matusita feature screening
- SHAP interpretation
- geemap.ml model back-write to GEE
- stricter post-processing and area reporting
        


In [ ]:
from pathlib import Path
import warnings

import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

OUTPUT_DIR = Path("outputs") / "wildapple_localrun_2021"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "gee_project": "kindle-400911",
    "year": 2021,
    "start_month": 3,
    "end_month": 10,
    "scale": 20,
    "random_seed": 42,
    "wildapple_asset": "projects/kindle-400911/assets/wildapple_sample",
    "worldcover_points_per_class": 150,
    "enable_wildapple_buffer": True,
    "wildapple_buffer_m": 20,
    "rect_sample_scale": 30,
    "wildapple_rect_group_id": "wildapple_rect_01",
    "wildapple_rect_corners": [
        [82.77484146, 43.20874077],
        [82.77116641, 43.2112944],
        [82.77223955, 43.21193612],
        [82.7756369, 43.20940169],
    ],
    "adm0_name": "China",
    "adm1_name": "Xinjiang Uygur Zizhiqu",
    "adm2_name": "Ili Kazakh",
    "valley_elevation_threshold_m": 1800,
    "valley_slope_threshold_deg": 12,
    "postprocess_min_patch_pixels": 150,
}

CLASS_INFO = {
    1: "Wild Apple Forest",
    2: "Other Forest",
    3: "Cropland",
    4: "Grassland/Shrub",
    5: "Urban/Bare",
    6: "Water/Snow/Ice",
}

MONTHS = list(range(CONFIG["start_month"], CONFIG["end_month"] + 1))
PROJ = ee.Projection("EPSG:3857").atScale(CONFIG["scale"])
OPTICAL_SERIES_PREFIXES = ["NDVI", "EVI", "NDMI", "LSWI"]
SAR_SERIES_PREFIXES = ["VV", "VH", "VV_VH_ratio", "VV_db", "VH_db"]
SUMMARY_PREFIXES = OPTICAL_SERIES_PREFIXES + SAR_SERIES_PREFIXES

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Months: {MONTHS}")
        


In [ ]:
ee.Initialize(project=CONFIG["gee_project"])
print("Earth Engine initialized.")
        


## 1. ROI and core utilities


In [ ]:
def get_ili_roi():
    admin = (
        ee.FeatureCollection("FAO/GAUL/2015/level2")
        .filter(ee.Filter.eq("ADM0_NAME", CONFIG["adm0_name"]))
        .filter(ee.Filter.eq("ADM1_NAME", CONFIG["adm1_name"]))
        .filter(ee.Filter.eq("ADM2_NAME", CONFIG["adm2_name"]))
    )
    return admin.geometry()


def month_tag(month):
    return f"M{int(month):02d}"


def month_band_names(prefix):
    return [f"{prefix}_{month_tag(month)}" for month in MONTHS]


def add_string_id(fc, id_field="sample_id", prefix="sample"):
    fc = ee.FeatureCollection(fc)
    size = fc.size()
    fc_list = fc.toList(size)

    def _mapper(i):
        i = ee.Number(i)
        feature = ee.Feature(fc_list.get(i))
        sample_id = ee.String(prefix).cat("_").cat(i.format("%05d"))
        return feature.set(id_field, sample_id)

    return ee.FeatureCollection(ee.List.sequence(0, size.subtract(1)).map(_mapper))


def build_rect_geometry(corners):
    return ee.Geometry.Polygon([corners], None, False)


ROI = get_ili_roi()
print("ROI ready.")

Map = geemap.Map()
Map.centerObject(ROI, 7)
Map.addLayer(ROI, {}, "ROI")
Map
        


## 2. Sentinel-2, Sentinel-1, terrain, and monthly feature image


In [ ]:
S2_RAW_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
S2_RENAMED_BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]
S2_QA_COLLECTION = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
S2_QA_BAND = "cs_cdf"
S2_CLEAR_THRESHOLD = 0.60


def add_optical_indices(image):
    ndvi = image.normalizedDifference(["nir", "red"]).rename("NDVI")
    evi = image.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {"NIR": image.select("nir"), "RED": image.select("red"), "BLUE": image.select("blue")},
    ).rename("EVI")
    ndmi = image.normalizedDifference(["nir", "swir1"]).rename("NDMI")
    lswi = image.normalizedDifference(["nir", "swir2"]).rename("LSWI")
    ndbi = image.normalizedDifference(["swir1", "nir"]).rename("NDBI")
    bsi = image.expression(
        "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
        {"SWIR": image.select("swir1"), "RED": image.select("red"), "NIR": image.select("nir"), "BLUE": image.select("blue")},
    ).rename("BSI")
    gcvi = image.expression("(NIR / GREEN) - 1", {"NIR": image.select("nir"), "GREEN": image.select("green")}).rename("GCVI")
    return image.addBands([ndvi, evi, ndmi, lswi, ndbi, bsi, gcvi])


def preprocess_s2(image):
    image = image.updateMask(image.select(S2_QA_BAND).gte(S2_CLEAR_THRESHOLD))
    image = image.select(S2_RAW_BANDS).rename(S2_RENAMED_BANDS).multiply(0.0001)
    return add_optical_indices(image).resample("bilinear").reproject(PROJ)


def load_monthly_s2(month):
    start_date = ee.Date.fromYMD(CONFIG["year"], month, 1)
    end_date = start_date.advance(1, "month")
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ROI)
        .filterDate(start_date, end_date)
        .linkCollection(S2_QA_COLLECTION, [S2_QA_BAND])
        .map(preprocess_s2)
    )
    return collection.median().setDefaultProjection(PROJ).clip(ROI)


def preprocess_s1(image):
    vv = image.select("VV")
    vh = image.select("VH")
    vv_db = ee.Image.constant(10).multiply(vv.log10()).rename("VV_db")
    vh_db = ee.Image.constant(10).multiply(vh.log10()).rename("VH_db")
    ratio = vv.divide(vh.max(ee.Image.constant(0.0001))).rename("VV_VH_ratio")
    return image.select(["VV", "VH"]).addBands([vv_db, vh_db, ratio]).resample("bilinear").reproject(PROJ)


def load_monthly_s1(month):
    start_date = ee.Date.fromYMD(CONFIG["year"], month, 1)
    end_date = start_date.advance(1, "month")
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(ROI)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .map(preprocess_s1)
    )
    return collection.median().setDefaultProjection(PROJ).clip(ROI)


def rename_with_month(image, month):
    suffix = month_tag(month)
    new_names = image.bandNames().map(lambda name: ee.String(name).cat("_").cat(suffix))
    return image.rename(new_names)


def build_monthly_feature_image():
    monthly_images = []
    for month in MONTHS:
        optical = load_monthly_s2(month)
        sar = load_monthly_s1(month)
        monthly_images.append(rename_with_month(ee.Image.cat([optical, sar]), month))
    return ee.Image.cat(monthly_images).clip(ROI)


def summary_from_month_stack(image, prefix):
    names = month_band_names(prefix)
    stack = image.select(names)
    stack_mean = stack.reduce(ee.Reducer.mean()).rename(f"{prefix}_mean")
    stack_std = stack.reduce(ee.Reducer.stdDev()).rename(f"{prefix}_std")
    stack_min = stack.reduce(ee.Reducer.min()).rename(f"{prefix}_min")
    stack_max = stack.reduce(ee.Reducer.max()).rename(f"{prefix}_max")
    stack_amp = stack_max.subtract(stack_min).rename(f"{prefix}_amp")

    array_image = stack.toArray()
    peak_index = array_image.arrayArgmax().arrayGet([0])
    peak_month = ee.Image(peak_index).add(CONFIG["start_month"]).rename(f"{prefix}_peak_month")

    first_band = stack.select([0]).rename(f"{prefix}_spring_ref")
    mid_band = stack.select([int(len(MONTHS) / 2)]).rename(f"{prefix}_summer_ref")
    last_band = stack.select([-1]).rename(f"{prefix}_autumn_ref")
    spring_rise = mid_band.subtract(first_band).rename(f"{prefix}_spring_rise")
    autumn_decline = mid_band.subtract(last_band).rename(f"{prefix}_autumn_decline")

    return ee.Image.cat([stack_mean, stack_std, stack_min, stack_max, stack_amp, peak_month, spring_rise, autumn_decline])


def build_summary_image(image):
    return ee.Image.cat([summary_from_month_stack(image, prefix) for prefix in SUMMARY_PREFIXES])


def build_terrain_image():
    dem = ee.Image("USGS/SRTMGL1_003").clip(ROI)
    terrain = ee.Algorithms.Terrain(dem)
    elevation = terrain.select("elevation").rename("elevation")
    slope = terrain.select("slope").rename("slope")
    aspect = terrain.select("aspect").rename("aspect")
    valley_proxy = elevation.lt(CONFIG["valley_elevation_threshold_m"]).And(slope.lt(CONFIG["valley_slope_threshold_deg"]))
    valley_distance = valley_proxy.fastDistanceTransform(128, "pixels", "squared_euclidean").sqrt().multiply(CONFIG["scale"]).rename("dist_to_valley_proxy_m")
    return ee.Image.cat([elevation, slope, aspect, valley_distance])


def build_texture_image(monthly_image):
    ndvi_name = "NDVI_M07" if 7 in MONTHS else f"NDVI_{month_tag(MONTHS[len(MONTHS) // 2])}"
    gray = monthly_image.select(ndvi_name).unitScale(-0.2, 0.8).multiply(100).toInt()
    glcm = gray.glcmTexture(size=3)
    selected = glcm.select([
        f"{ndvi_name}_contrast",
        f"{ndvi_name}_diss",
        f"{ndvi_name}_ent",
        f"{ndvi_name}_idm",
        f"{ndvi_name}_asm",
        f"{ndvi_name}_var",
    ])
    return selected.rename(["NDVI_tex_contrast", "NDVI_tex_diss", "NDVI_tex_ent", "NDVI_tex_idm", "NDVI_tex_asm", "NDVI_tex_var"])


MONTHLY_IMAGE = build_monthly_feature_image()
SUMMARY_IMAGE = build_summary_image(MONTHLY_IMAGE)
TERRAIN_IMAGE = build_terrain_image()
TEXTURE_IMAGE = build_texture_image(MONTHLY_IMAGE)
FEATURE_IMAGE = ee.Image.cat([MONTHLY_IMAGE, SUMMARY_IMAGE, TERRAIN_IMAGE, TEXTURE_IMAGE]).clip(ROI)

print("Feature image band count:", FEATURE_IMAGE.bandNames().size().getInfo())
print("First 20 bands:", FEATURE_IMAGE.bandNames().getInfo()[:20])
        


## 3. Sample loading, optional wild apple buffer expansion, and GEE sampling


In [ ]:
def load_wildapple_samples():
    fc = ee.FeatureCollection(CONFIG["wildapple_asset"]).filterBounds(ROI)
    fc = add_string_id(fc, id_field="sample_id", prefix="wildapple")
    return fc.map(lambda f: ee.Feature(f).set({"class": 1, "source": "wildapple", "group_id": f.get("sample_id")}))


def build_rect_wildapple_samples():
    rect = build_rect_geometry(CONFIG["wildapple_rect_corners"])
    rect_fc = ee.Image.pixelLonLat().sample(
        region=rect,
        scale=CONFIG["rect_sample_scale"],
        geometries=True,
    )
    rect_fc = add_string_id(rect_fc, id_field="sample_id", prefix="wildapple_rect")
    return rect_fc.map(
        lambda f: ee.Feature(f).set({
            "class": 1,
            "source": "wildapple_rect",
            "group_id": CONFIG["wildapple_rect_group_id"],
        })
    )


def build_worldcover_samples():
    worldcover = ee.ImageCollection("ESA/WorldCover/v200")
    wc_2021 = ee.Image(
        ee.Algorithms.If(
            worldcover.filter(ee.Filter.eq("YEAR", 2021)).size().gt(0),
            worldcover.filter(ee.Filter.eq("YEAR", 2021)).first(),
            ee.Algorithms.If(
                worldcover.filter(ee.Filter.eq("year", 2021)).size().gt(0),
                worldcover.filter(ee.Filter.eq("year", 2021)).first(),
                worldcover.first(),
            ),
        )
    ).select("Map")

    wc_to_class = wc_2021.remap([10, 20, 30, 40, 50, 60, 70, 80, 90], [2, 4, 4, 3, 5, 5, 6, 6, 4]).rename("class").clip(ROI)
    fc = wc_to_class.stratifiedSample(
        numPoints=CONFIG["worldcover_points_per_class"] * 5,
        classBand="class",
        region=ROI,
        scale=CONFIG["scale"],
        tileScale=4,
        geometries=True,
        classValues=[2, 3, 4, 5, 6],
        classPoints=[CONFIG["worldcover_points_per_class"]] * 5,
    )
    fc = add_string_id(fc, id_field="sample_id", prefix="worldcover")
    return fc.map(lambda f: ee.Feature(f).set({"source": "worldcover", "group_id": f.get("sample_id")}))


def maybe_expand_wildapple_regions(fc):
    if not CONFIG["enable_wildapple_buffer"]:
        return fc
    return fc.map(lambda f: ee.Feature(f.geometry().buffer(CONFIG["wildapple_buffer_m"]), f.toDictionary()))


def build_training_sample_fc(feature_image):
    wildapple_points = load_wildapple_samples().merge(build_rect_wildapple_samples())
    wildapple_regions = maybe_expand_wildapple_regions(wildapple_points)
    worldcover_points = build_worldcover_samples()
    sampling_regions = worldcover_points.merge(wildapple_regions)
    return feature_image.sampleRegions(
        collection=sampling_regions,
        properties=["class", "source", "sample_id", "group_id"],
        scale=CONFIG["scale"],
        tileScale=4,
        geometries=True,
    )


SAMPLE_FC = build_training_sample_fc(FEATURE_IMAGE)
print("Sample feature count:", SAMPLE_FC.size().getInfo())
        


## 4. Local baseline table and grouped Random Forest


In [ ]:
df_raw = geemap.ee_to_df(SAMPLE_FC)
df_raw.to_csv(OUTPUT_DIR / "baseline_samples_raw.csv", index=False)
print(df_raw.shape)
df_raw.head()
        


In [ ]:
NON_FEATURE_COLUMNS = {"class", "source", "sample_id", "group_id", "system:index", ".geo"}


def prepare_baseline_table(df):
    df = df.copy()
    feature_cols = [col for col in df.columns if col not in NON_FEATURE_COLUMNS]
    numeric_features = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    valid_mask = df["class"].notna() & df["group_id"].notna()
    numeric_features = numeric_features.loc[valid_mask]
    target = df.loc[valid_mask, "class"].astype(int)
    groups = df.loc[valid_mask, "group_id"].astype(str)
    return numeric_features, target, groups, feature_cols


X, y, groups, feature_cols = prepare_baseline_table(df_raw)
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols, index=X.index)

gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=CONFIG["random_seed"])
train_idx, test_idx = next(gss.split(X_imputed, y, groups))

X_train = X_imputed.iloc[train_idx]
X_test = X_imputed.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=CONFIG["random_seed"],
    n_jobs=-1,
    class_weight="balanced_subsample",
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
oa = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Overall accuracy: {oa:.4f}")
print(f"Kappa: {kappa:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

report_df = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T
report_df.to_csv(OUTPUT_DIR / "baseline_classification_report.csv")

label_order = sorted(CLASS_INFO)
label_names = [CLASS_INFO[label] for label in label_order]
cm = confusion_matrix(y_test, y_pred, labels=label_order)
cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
cm_df.to_csv(OUTPUT_DIR / "baseline_confusion_matrix.csv")

plt.figure(figsize=(9, 7))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Baseline confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "baseline_confusion_matrix.png", dpi=200)
plt.show()

feature_importance_df = (
    pd.DataFrame({"feature": feature_cols, "importance": rf.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
feature_importance_df.to_csv(OUTPUT_DIR / "baseline_feature_importance.csv", index=False)

top_n = 30 if len(feature_importance_df) > 30 else len(feature_importance_df)
plt.figure(figsize=(10, 10))
sns.barplot(data=feature_importance_df.head(top_n), y="feature", x="importance", palette="viridis")
plt.title("Baseline feature importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "baseline_feature_importance.png", dpi=200)
plt.show()

feature_importance_df.head(20)
        


## 5. Helpers for the next stage: long time series table, cubic fit, and JM distance

These helpers are not fully wired into the baseline model yet.
They are included now so the notebook structure already matches the refactor plan.
        


In [ ]:
def build_long_timeseries_table(df, prefixes=None):
    prefixes = prefixes or (OPTICAL_SERIES_PREFIXES + ["VV", "VH", "VV_VH_ratio"])
    rows = []
    id_cols = ["class", "source", "sample_id", "group_id"]

    for prefix in prefixes:
        value_cols = [col for col in df.columns if col.startswith(f"{prefix}_M")]
        if not value_cols:
            continue
        melted = df[id_cols + value_cols].melt(
            id_vars=id_cols,
            value_vars=value_cols,
            var_name="band_month",
            value_name="value",
        )
        melted["feature_name"] = prefix
        melted["month"] = melted["band_month"].str.extract(r"M(\d{2})").astype(int)
        melted["time_offset"] = melted["month"] - CONFIG["start_month"]
        rows.append(melted)

    if not rows:
        return pd.DataFrame()

    long_df = pd.concat(rows, ignore_index=True)
    long_df["class_name"] = long_df["class"].map(CLASS_INFO)
    return long_df.sort_values(["feature_name", "group_id", "month"]).reset_index(drop=True)


def fit_cubic_series(group):
    x = group["time_offset"].to_numpy(dtype=float)
    y_values = pd.to_numeric(group["value"], errors="coerce").to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y_values)
    x = x[mask]
    y_values = y_values[mask]
    result = {
        "group_id": group["group_id"].iloc[0],
        "sample_id": group["sample_id"].iloc[0],
        "class": group["class"].iloc[0],
        "class_name": group["class_name"].iloc[0],
        "feature_name": group["feature_name"].iloc[0],
    }
    if len(x) < 4:
        result.update({"a3": np.nan, "a2": np.nan, "a1": np.nan, "a0": np.nan, "r2": np.nan})
        return result
    coeffs = np.polyfit(x, y_values, deg=3)
    pred = np.polyval(coeffs, x)
    ss_res = np.sum((y_values - pred) ** 2)
    ss_tot = np.sum((y_values - np.mean(y_values)) ** 2)
    result.update({
        "a3": coeffs[0],
        "a2": coeffs[1],
        "a1": coeffs[2],
        "a0": coeffs[3],
        "r2": np.nan if ss_tot == 0 else 1 - ss_res / ss_tot,
    })
    return result


def batch_fit_cubic(long_df):
    if long_df.empty:
        return pd.DataFrame()
    grouped = long_df.groupby(["feature_name", "group_id"], sort=False)
    return pd.DataFrame([fit_cubic_series(group) for _, group in grouped])


def jeffries_matusita_distance(x1, x2):
    x1 = pd.to_numeric(pd.Series(x1), errors="coerce").dropna().to_numpy(dtype=float)
    x2 = pd.to_numeric(pd.Series(x2), errors="coerce").dropna().to_numpy(dtype=float)
    if len(x1) < 2 or len(x2) < 2:
        return np.nan
    m1, m2 = np.mean(x1), np.mean(x2)
    v1, v2 = np.var(x1, ddof=1), np.var(x2, ddof=1)
    if v1 <= 0 or v2 <= 0:
        return np.nan
    bhattacharyya = ((m1 - m2) ** 2) / (4 * (v1 + v2)) + 0.5 * np.log((v1 + v2) / (2 * np.sqrt(v1 * v2)))
    return float(2 * (1 - np.exp(-bhattacharyya)))


LONG_DF = build_long_timeseries_table(df_raw)
FITTED_CUBIC_DF = batch_fit_cubic(LONG_DF)

LONG_DF.to_csv(OUTPUT_DIR / "monthly_long_table.csv", index=False)
FITTED_CUBIC_DF.to_csv(OUTPUT_DIR / "cubic_fit_table.csv", index=False)

print("Long monthly table:", LONG_DF.shape)
print("Cubic fit table:", FITTED_CUBIC_DF.shape)
FITTED_CUBIC_DF.head()
        


## 6. Area statistics helper for the later GEE classification image


In [ ]:
def calculate_area_by_class(classified_image, roi, class_info):
    pixel_area = ee.Image.pixelArea().rename("area")
    features = []
    for class_id, class_name in class_info.items():
        class_mask = classified_image.eq(class_id)
        area_m2 = pixel_area.updateMask(class_mask).reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=roi,
            scale=CONFIG["scale"],
            maxPixels=1e13,
        ).get("area")
        features.append(ee.Feature(None, {"class_id": class_id, "class_name": class_name, "area_m2": area_m2}))
    return ee.FeatureCollection(features)


# TODO in the next implementation step:
# 1. convert sklearn RF to a GEE classifier with geemap.ml
# 2. classify FEATURE_IMAGE in GEE
# 3. apply conservative smoothing and patch filtering
# 4. run calculate_area_by_class on the post-processed image
        


## 7. Next implementation checkpoints

Recommended next actions in this notebook:
1. replace the wild apple asset path and verify the baseline sample table
2. add sample cleaning rules and outlier filters
3. merge cubic-fit coefficients back into the modeling table
4. run Jeffries-Matusita + RF screening on candidate features
5. add SHAP global and local plots
6. back-write the selected RF into GEE and finish map smoothing + area reports
        
